## Lesson 5: Advanced Python - Cleaning Dirty Data with Regex and Fuzzy Matching

Today is about the "Janitor Work" that makes the "Analysis Work" possible.

A 1x analyst complains that the data is "trash." A 100x analyst treats dirty data as a competitive advantage. If you can clean what others can't, you find insights they miss.

#### The Problem: The "Data Entropy"
In our fraud case, users (and even our own staff) enter codes in different formats.

- Database says: SPRING20, spring20, SPRING20, Spring_20, SPRNG20.
- SQL sees: 5 different codes.
- Reality: It’s 1 code being used 5 different ways.

If your script only looks for SPRING20, you are blind to 80% of the activity.

#### Phase 1: The SQL "First Pass" (Standardization)
Before pulling data into Python, we use SQL to "normalize" the easy stuff. This saves processing power later.

**The 100x SQL Pattern:**
We use TRIM, UPPER, and REGEXP_REPLACE (or REPLACE) to strip the noise.

In [ ]:
SELECT 
    -- 1. Remove leading/trailing spaces
    -- 2. Force Uppercase
    -- 3. Remove non-alphanumeric characters (hyphens, underscores)
    UPPER(TRIM(REPLACE(REPLACE(discount_code, '-', ''), '_', ''))) as clean_code,
    COUNT(*) as usage_count
FROM order_discounts
GROUP BY 1
ORDER BY usage_count DESC;

#### Phase 2: The Python "Deep Clean" (Fuzzy Matching)
SQL is great for fixed rules, but it's terrible at Typos. If a user types SPRNG20 (missing the 'I'), SQL's REPLACE won't save you.
This is where we use Levenshtein Distance, a mathematical way to measure how many "edits" it takes to turn one string into another.

**The Execution:**
We use a library like thefuzz (formerly FuzzyWuzzy).

In [ ]:
from thefuzz import process, fuzz

# Our "Source of Truth" (Valid codes from Marketing)
valid_codes = ["SPRING20", "WELCOME10", "BLACKFRIDAY"]

# Dirty data from our database
dirty_codes = ["SPRNG20", "spring 20", "WLCME10", "B-FRIDAY"]

def clean_and_match(code):
    # 1. Basic Python Cleanup
    normalized = code.strip().upper().replace("-", "").replace(" ", "")
    
    # 2. Check for exact match
    if normalized in valid_codes:
        return normalized
    
    # 3. Fuzzy Match (The 100x Magic)
    # This finds the best match in our list with a score
    best_match, score = process.extractOne(normalized, valid_codes)
    
    # If the match is > 80% identical, we assume it's a typo
    if score > 80:
        return best_match
    
    return "UNKNOWN_CODE"

# Testing it out
for code in dirty_codes:
    print(f"Original: {code} -> Resolved To: {clean_and_match(code)}")

#### The 100x Solution Provisioning: "The Mapping Table"
Don't just run this script every time. That’s inefficient.
The 100x Strategy: Create a Mapping Table in your database.

- Run the Python script once.
- Store the results: Original: SPRNG20 | Resolved: SPRING20.
- In your SQL reports, JOIN your orders to this mapping table.

**The Delivery to your Manager:**
"I noticed our discount reporting was under-counting usage by 15% due to manual entry typos. I’ve implemented a fuzzy-matching logic to group these variants, ensuring our ROI calculations for marketing campaigns are now 100% accurate."

#### Your 10-Minute Mission
You have a list of city_names: ["New York", "New-York", "NY", "N.Y.", "Big Apple"].

- Regex/Standardization: Which of these can you fix with simple SQL REPLACE?
- Fuzzy Matching: Which of these would fail fuzzy matching against "New York" even with a high score? (Hint: Think about "NY" vs "New York").
- The 100x fix: How would you handle "NY"?

#### 1x Analyst:
You can fix New-York wit a simple regex that removes the -
N.Y., NY and Big apple will fail fuzzy matching against New York
For NY, it might be better to have a table that contains the short form of the city and check against it


You are thinking like an Analytics Engineer now. You’ve identified the limit of Fuzzy Logic and the necessity of Master Data Management (MDM).

1. The Logic Wall
You’re 100% correct.

Fuzzy Matching relies on character distance. "NY" (2 letters) vs "New York" (8 letters) has a very low similarity score because they share almost no common sequence.

Big Apple is a semantic alias—it has 0% character similarity. No algorithm in the world will "guess" that without a reference.

2. The 100x Solution: The "Aliasing" Table
Instead of writing a 50-line CASE WHEN statement in every SQL script, a 100x analyst creates a Canonical Mapping Table.

The SQL Structure:
| raw_input (Key) | canonical_name (Value) | type |
| :--- | :--- | :--- |
| NY | New York | Abbreviation |
| N.Y. | New York | Abbreviation |
| Big Apple | New York | Alias |
| New-York | New York | Regex-Fix |
| SPRNG20 | SPRING20 | Fuzzy-Typo |

The 100x Workflow:

- Join: Always LEFT JOIN your raw data to this mapping table.
- Coalesce: Use COALESCE(map.canonical_name, raw.city_name) so that if a new city appears that isn't in your map yet, the data doesn't disappear—it just stays in its raw form until you "map" it.

- Communicating the "Janitor" Work
When you do this, don't tell your CEO: I fixed the city names. That sounds small.

3. The 100x Delivery:
I’ve implemented a Master Data Layer that centralizes our geographic and promotional logic. This eliminates 'Data Silos' and ensures that whether a customer types 'NY' or 'New York', our executive dashboard reflects a single, unified truth for that region.